In [ ]:
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt

# Describir las tablas existentes sin alterar sus datos.
timesheet_path = Path("timesheet.csv")
drivers_path = Path("drivers.csv")

if not timesheet_path.exists() or not drivers_path.exists():
    search_dirs = [Path.cwd(), *Path.cwd().parents]
    data_dirs = ("", "data", "datasets")

    for filename, path_var in (
        ("timesheet.csv", "timesheet_path"),
        ("drivers.csv", "drivers_path"),
    ):
        current_path = globals()[path_var]
        if current_path.exists():
            continue

        found = next(
            (
                base / subdir / filename
                for base in search_dirs
                for subdir in data_dirs
                if (base / subdir / filename).is_file()
            ),
            None,
        )
        if found is not None:
            if path_var == "timesheet_path":
                timesheet_path = found
            else:
                drivers_path = found
        else:
            raise FileNotFoundError(
                f"Could not find {filename}. Notebook working directory: "
                f"{Path.cwd()}. Place the file there or in a parent "
                "directory's data/ or datasets/ folder."
            )

timesheet = pd.read_csv(timesheet_path)
drivers = pd.read_csv(drivers_path)

for table_name, table in [("timesheet", timesheet), ("drivers", drivers)]:
    print(f"\n{table_name}:")
    display(pd.DataFrame({
        "columna": table.columns,
        "tipo": [str(dtype) for dtype in table.dtypes]
    }))

def find_driver_key(table):
    for candidate in ("driver_id", "driverId"):
        if candidate in table.columns:
            return candidate
    raise KeyError("No se encontró driver_id ni driverId.")

timesheet_key = find_driver_key(timesheet)
drivers_key = find_driver_key(drivers)

required_timesheet = {timesheet_key, "hours-logged", "miles-logged"}
required_drivers = {drivers_key, "name"}
if missing := required_timesheet - set(timesheet.columns):
    raise KeyError(f"Faltan columnas en timesheet: {missing}")
if missing := required_drivers - set(drivers.columns):
    raise KeyError(f"Faltan columnas en drivers: {missing}")

print(
    "\nLlave de relación: identificador del conductor "
    f"({timesheet_key} en timesheet, {drivers_key} en drivers)."
)
print(
    "Plan: resumir horas y millas por conductor, comparar horas con su "
    "promedio, unir el resumen con los nombres, seleccionar los diez "
    "conductores con más millas y guardar la tabla y el gráfico."
)

# Promedios por conductor; no se incluye week.
mean_timesheet = (
    timesheet.groupby(timesheet_key, as_index=False)
    .agg({
        "hours-logged": "mean",
        "miles-logged": "mean"
    })
)

# Copia de timesheet con el promedio de horas por conductor.
timesheet_with_means = timesheet.copy()
timesheet_with_means["mean_hours-logged"] = (
    timesheet.groupby(timesheet_key)["hours-logged"].transform("mean")
)

# Registros con horas por debajo del promedio del conductor.
timesheet_below = timesheet_with_means[
    timesheet_with_means["hours-logged"]
    < timesheet_with_means["mean_hours-logged"]
].copy()

# Sumas y mínimo/máximo de horas por conductor.
sum_timesheet = (
    timesheet.groupby(timesheet_key, as_index=False)
    .agg({
        "hours-logged": "sum",
        "miles-logged": "sum"
    })
)

min_max_timesheet = (
    timesheet.groupby(timesheet_key, as_index=False)
    .agg(
        min_hours_logged=("hours-logged", "min"),
        max_hours_logged=("hours-logged", "max")
    )
)

# Unir nombres y dejar el identificador con el nombre indicado en la observación.
drivers_for_join = drivers[[drivers_key, "name"]].copy()
if drivers_key != timesheet_key:
    drivers_for_join = drivers_for_join.rename(columns={drivers_key: timesheet_key})

summary = sum_timesheet.merge(
    drivers_for_join,
    on=timesheet_key,
    how="left",
    validate="many_to_one"
)
if timesheet_key != "driver_id":
    summary = summary.rename(columns={timesheet_key: "driver_id"})

summary = summary[["driver_id", "hours-logged", "miles-logged", "name"]]
summary.to_csv("summary.csv", index=False)

# Diez conductores con más millas y gráfico con el mayor en la parte superior.
top10 = summary.nlargest(10, "miles-logged").sort_values(
    "miles-logged", ascending=False
)

ax = top10.plot(
    kind="barh",
    x="name",
    y="miles-logged",
    legend=False,
    figsize=(10, 6)
)
ax.set_xlabel("Millas registradas")
ax.set_ylabel("Conductor")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("top10_drivers.png", dpi=300, bbox_inches="tight")
plt.show()

FileNotFoundError: Place timesheet.csv and drivers.csv in the notebook's working directory or update their paths.